# ViT Transformer Block（Vision Transformer 块）

**难度：** Hard

**函数签名：** `ViTBlock(d_model, num_heads)`（nn.Module）

**前向传播：** `forward(x) -> Tensor`
- `x` — 输入张量 (B, N, d_model)，N = 图像块数 + 1（含 CLS token）

**返回：** 输出张量 (B, N, d_model)

**架构：**
```
x = x + MHA(LayerNorm(x))
x = x + MLP(LayerNorm(x))
```

**约束：**
- `nn.Linear` 和 `nn.LayerNorm` 可作为构建块
- MHA 必须手动实现（不得使用 `nn.MultiheadAttention`）
- MLP：`Linear(d, 4d) -> GELU -> Linear(4d, d)`
- GELU 必须手动实现：`x * 0.5 * (1 + erf(x / sqrt(2)))`
- 不得调用 `F.*` 或 `nn.functional.*`

**提示：** MHA 用 `nn.Linear(d, 3d)` 同时投影 Q/K/V，GELU 用 `torch.erf` 实现

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✅ SOLUTION

class ViTBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        # d_h: 每个注意力头的维度 = d_model / num_heads
        # 例: d_model=64, num_heads=4 → d_h=16
        self.d_h = d_model // num_heads

        # ---- Pre-Norm 层 ----
        # ViT 使用 pre-norm 架构：先 LayerNorm 再做子层操作
        # 这比 post-norm 训练更稳定
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # ---- MHA 参数 ----
        # 用一个大 Linear 同时投影 Q, K, V，比分成三个 Linear 更高效
        # 输出维度 3*d_model，后面拆分成 Q, K, V 各 d_model
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        # 输出投影：将多头拼接后的结果投影回 d_model 维
        self.proj = nn.Linear(d_model, d_model)

        # ---- MLP 参数 ----
        # 标准 ViT MLP: d_model → 4*d_model → d_model
        # 先升维 4 倍再降回来，增加非线性表达能力
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)

    def _gelu(self, x):
        # ---- 手写 GELU 激活函数 ----
        # GELU(x) = x * Φ(x)，其中 Φ 是标准正态的 CDF
        # 精确公式: x * 0.5 * (1 + erf(x / √2))
        # erf 是误差函数，torch.erf 直接可用
        # 为什么用 GELU 而不是 ReLU: GELU 是平滑的，梯度不会在 0 处突变
        return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

    def _mha(self, x):
        # ---- 多头自注意力 ----
        B, N, D = x.shape

        # Step 1: 投影 QKV
        # qkv 形状: (B, N, 3*D) — Q, K, V 拼在一起
        # reshape → (B, N, 3, num_heads, d_h) — 拆分出 Q/K/V 和多头
        # permute → (3, B, num_heads, N, d_h) — 把 3 放到最前面方便拆分
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.d_h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        # q, k, v 各自形状: (B, num_heads, N, d_h)

        # Step 2: 缩放点积注意力
        # scale = 1/√d_h，防止点积值过大导致 softmax 饱和
        scale = self.d_h ** -0.5
        # q @ k.T: (B, H, N, d_h) @ (B, H, d_h, N) → (B, H, N, N)
        # 每个 (i,j) 位置是 query_i 和 key_j 的点积相似度
        attn = torch.softmax(q @ k.transpose(-2, -1) * scale, dim=-1)
        # attn @ v: (B, H, N, N) @ (B, H, N, d_h) → (B, H, N, d_h)
        # 按注意力权重对 value 加权求和

        # Step 3: 合并多头 + 输出投影
        # transpose(1,2): (B, H, N, d_h) → (B, N, H, d_h)
        # reshape: (B, N, H*d_h) = (B, N, D) — 多头拼接
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(out)

    def forward(self, x):
        # ---- Pre-Norm + 残差连接 ----
        # 第一个子层: 自注意力
        # norm1(x) → _mha → 残差加回 x
        x = x + self._mha(self.norm1(x))
        # 第二个子层: MLP
        # norm2(x) → fc1 → GELU → fc2 → 残差加回 x
        # 注意 GELU 夹在两个 Linear 之间
        x = x + self.fc2(self._gelu(self.fc1(self.norm2(x))))
        return x

In [ ]:
torch.manual_seed(0)
batch, num_patches, d_model, num_heads = 2, 16, 64, 4

block = ViTBlock(d_model, num_heads)
x = torch.randn(batch, num_patches, d_model)
out = block(x)

print("Input shape: ", x.shape)    # (2, 16, 64)
print("Output shape:", out.shape)  # (2, 16, 64)
assert out.shape == x.shape, "Shape mismatch!"
print("Shape preserved: True")

In [ ]:
from torch_judge import check
check("vit_block")

## 📝 核心思路总结

1. **Pre-Norm 架构**：先 LayerNorm 再做子层，比 Post-Norm 训练更稳定
2. **多头注意力**：QKV 用一个大 Linear 投影后 reshape 拆分，效率更高
3. **手写 GELU**：`x * 0.5 * (1 + erf(x/√2))`，面试高频手撕题
4. **残差连接**：`x = x + sublayer(x)`，保证梯度流动